# Chapitre 4 : Exploration structurelle et Indexation avancée
## Audit de santé de la donnée et requêtage ciblé en mémoire

Avant toute transformation ou nettoyage, le Data Engineer doit inspecter la volumétrie, valider la complétude des colonnes et extraire des sous-ensembles précis. Deux compétences sont indispensables :
1. **Le profiling rapide** : identifier les anomalies, les valeurs manquantes et les déséquilibres de distribution.
2. **L'indexation rigoureuse** : maîtriser la différence absolue entre `.loc` et `.iloc`, et formuler des masques booléens sans ambiguïté.

In [ ]:
import pandas as pd
import numpy as np

print("Pandas importé avec succès !")

## 1. Génération du dataset d'audit
Exécutez la cellule suivante pour générer une table d'activité de comptes utilisateurs (1000 lignes) contenant des valeurs manquantes, des statuts variés et des anomalies.

In [ ]:
np.random.seed(42)
n = 1000

data_users = {
    "user_id": [f"USR-{i:04d}" for i in range(1, n + 1)],
    "pays": np.random.choice(["FR", "BE", "CH", "DE", None], size=n, p=[0.5, 0.2, 0.1, 0.1, 0.1]),
    "score_activite": np.random.normal(loc=50, scale=15, size=n).round(1),
    "nb_connexions": np.random.randint(0, 45, size=n),
    "abonnement": np.random.choice(["FREE", "PRO", "ENTERPRISE"], size=n, p=[0.7, 0.2, 0.1])
}

df_users = pd.DataFrame(data_users)
# Injection artificielle de doublons et anomalies
df_users.loc[5:7, "score_activite"] = np.nan
df_users.loc[12, "score_activite"] = -999.0  # Valeur aberrante sentinelle

df_users.head()

### 🧪 Exercice 1 : Profiling et métriques de santé
Complétez la fonction `profiler_dataset(df)` ci-dessous pour qu'elle renvoie un dictionnaire contenant les indicateurs clés de qualité du dataset :
- `total_lignes` : nombre total de lignes.
- `colonnes_avec_nulls` : liste des noms de colonnes contenant au moins une valeur nulle (`NaN`).
- `taux_free` : proportion (entre 0 et 1) d'utilisateurs ayant l'abonnement `'FREE'` (utilisez `value_counts(normalize=True)`).

In [ ]:
def profiler_dataset(df: pd.DataFrame) -> dict:
    # TODO : Implémentez les calculs
    total_lignes = ...
    null_cols = ...  # Indice: df.columns[df.isna().any()].tolist()
    prop_free = ...  # Indice: df['abonnement'].value_counts(normalize=True)['FREE']
    
    return {
        "total_lignes": total_lignes,
        "colonnes_avec_nulls": null_cols,
        "taux_free": prop_free
    }

rapport = profiler_dataset(df_users)
print("Rapport d'audit :", rapport)

In [ ]:
assert rapport["total_lignes"] == 1000, "Le total de lignes doit être égal à 1000."
assert set(rapport["colonnes_avec_nulls"]) == {"pays", "score_activite"}, "Vérifiez la détection des colonnes nulles."
assert 0.65 <= rapport["taux_free"] <= 0.75, "Le taux de FREE doit être proche de 0.70."
print("✅ Exercice 1 validé avec succès !")

## 2. Maîtriser la sélection : `.loc` vs `.iloc`
Modifions l'index de `df_users` pour utiliser `user_id` comme clé primaire explicite :

In [ ]:
df_indexed = df_users.set_index("user_id")
df_indexed.head(3)

### 🧪 Exercice 2 : Extractions ciblées
À partir du DataFrame `df_indexed` :
1. Utilisez **`.iloc`** pour extraire les **5 premières lignes** physiques et les **2 premières colonnes** dans `extrait_iloc`.
2. Utilisez **`.loc`** pour extraire les utilisateurs de `'USR-0010'` à `'USR-0015'` (inclus) pour les colonnes `'pays'` et `'score_activite'` dans `extrait_loc`.

In [ ]:
# TODO : Extraction avec iloc (position)
extrait_iloc = df_indexed.iloc[..., ...]

# TODO : Extraction avec loc (labels)
extrait_loc = df_indexed.loc[..., ...]

print("Extrait iloc :\n", extrait_iloc)
print("\nExtrait loc :\n", extrait_loc)

In [ ]:
assert extrait_iloc.shape == (5, 2), f"Attendu (5, 2), obtenu {extrait_iloc.shape} pour iloc."
assert extrait_loc.shape == (6, 2), f"Attendu (6, 2), obtenu {extrait_loc.shape} pour loc (bornes incluses !)."
assert list(extrait_loc.columns) == ["pays", "score_activite"], "Les colonnes sélectionnées via loc sont incorrectes."
assert "USR-0015" in extrait_loc.index, "USR-0015 doit être présent dans l'index de extrait_loc."
print("✅ Exercice 2 validé avec succès !")

---
# 🏆 Défi de fin de séance (Cours 1) : L'Enquête Data

### Contexte métier
L'équipe produit souhaite identifier les comptes à fort potentiel pour une campagne de fidélisation ciblée.

**Consignes :**
À partir du DataFrame initial `df_users` :
1. Définissez un filtre booléen strict retenant les utilisateurs qui remplissent **toutes** les conditions suivantes :
   - Le pays fait partie de la zone francophone prioritaire : `'FR'`, `'BE'` ou `'CH'` (utilisez `.isin(...)`).
   - Le nombre de connexions est supérieur ou égal à 20 (`nb_connexions >= 20`).
   - Le score d'activité est strictement positif et non nul (`score_activite > 0`).
   - L'abonnement n'est **pas** un compte gratuit (c'est-à-dire `abonnement != 'FREE'`).
2. Stockez le résultat filtré dans un nouveau DataFrame nommé `df_cibles`.
3. Réinitialisez l'index pour obtenir une séquence propre de 0 à $N-1$ (`reset_index(drop=True)`).
4. Triez ce DataFrame par `score_activite` décroissant.

In [ ]:
# TODO : Construction du masque booléen combiné
masque = (
    df_users["pays"].isin(...) &
    (df_users["nb_connexions"] >= ...) &
    (df_users["score_activite"] > ...) &
    (df_users["abonnement"] != ...)
)

# Application du filtre, tri et réindexation
df_cibles = df_users[masque].copy()
df_cibles = df_cibles.sort_values(by=..., ascending=False).reset_index(drop=True)

print(f"Nombre de comptes ciblés : {len(df_cibles)}")
df_cibles.head()

In [ ]:
assert isinstance(df_cibles, pd.DataFrame), "df_cibles doit être un DataFrame."
assert not df_cibles.empty, "Le DataFrame df_cibles ne doit pas être vide !"
assert (df_cibles["abonnement"] != "FREE").all(), "Des comptes FREE subsistent dans le résultat."
assert (df_cibles["nb_connexions"] >= 20).all(), "Certains comptes ont moins de 20 connexions."
assert (df_cibles["score_activite"] > 0).all(), "Des scores d'activité négatifs ou nuls subsistent."
assert df_cibles["pays"].isin(["FR", "BE", "CH"]).all(), "Des pays hors zone FR/BE/CH sont présents."
assert df_cibles["score_activite"].is_monotonic_decreasing, "df_cibles doit être trié par score_activite décroissant !"
assert df_cibles.index[0] == 0 and df_cibles.index[-1] == len(df_cibles) - 1, "L'index doit être réinitialisé de 0 à N-1."

print("🎉 EXCELLENT TRAVAIL ! Tous les tests du Chapitre 4 et du Cours 1 sont validés !")